# StoCFL: efeito do dropout com retorno na clusterização

Compara a **clusterização** do StoCFL (`StoCFL-FCC+CFL.py --train 0`) em Rotated MNIST **com e sem dropout com retorno** (`--drop`, `--drop_rounds`), em várias seeds, e mostra matrizes de confusão e gráficos em Plotly. O desenho do dropout está no [plano](../../docs/plano-dropout-stocfl.md) e em D-032 de [decisoes.md](../../docs/decisoes.md); o notebook é o passo 4 do plano.

- **O que é o dropout aqui:** a cada rodada, cada cliente disponível **cai** com probabilidade `d` (`--drop`) e fica fora `L` rodadas, com `L` sorteado uniformemente em `{1, ..., 10}` (`--drop_rounds 10`; o mínimo é sempre 1). Na rodada seguinte à ausência ele volta e não pode cair de novo. Quem está fora **não é sorteado**: os `m` clientes da rodada são sorteados só entre os disponíveis (todos, se houver menos). Com `d = 0.1` e `L_max = 10`, a disponibilidade média é 0,645; com `d = 0.05`, 0,784 (`1 / (1 + d·5,5)`).
- **Cenários:** `StoCFL sem dropout` (`d = 0`, o sorteio de sempre) e `StoCFL com dropout` (`d = 0.05` e `d = 0.1`), para cada `m` em `FRACOES` (100% e 50% dos 240 clientes) e cada `k` em `KS` (8 e 16 grupos reais). 10 seeds (`0` a `9`), 50 rodadas, `τ = 0.75`.
- **Gabarito:** os grupos reais são rotações em `k` ângulos igualmente espaçados (`k = 8`: 0°, 45°, ...; `k = 16`: 0°, 22,5°, ...), e o cliente `c` pertence ao grupo `c * k // N`.
- **Sem treino:** só a observação de `Ψ` e a fusão de clusters; o dropout atrasa a observação dos clientes e reduz quantos entram na matriz de cosseno a cada rodada.
- **Como roda:** cada combinação chama o script por `subprocess`, sem alterar o código do StoCFL, e o notebook lê o `cluster.pkl`, o `availability.pkl` e o log de cada execução. Execuções concluídas com os mesmos parâmetros (inclusive `τ`) são reaproveitadas. Com `DADOS_JSON = True`, o notebook não lê logs nem roda nada: carrega tudo de `logs/resultados-dropout.json` e só plota, o que permite apagar as pastas de `logs/` depois de gravar o JSON. Sem as execuções e com o JSON desta grade já gravado, ele para com um aviso em vez de rodar tudo de novo (`REEXECUTAR = True` força a nova execução).
- **Ambiente:** o kernel é o `.venv` da raiz (ver o README da raiz); abra o notebook a partir de `projetos/StoCFL/`. Na primeira execução, cada seed gera partições em `datasets/rotated_mnist_240_<seed>_k<k>/`.

**Custo da grade completa** (2 `k` × 2 `m` × 3 cenários × 10 seeds = **120 execuções**): cada execução grava `psi.pkl` (~1,6 GB com 240 clientes) a cada rodada em `logs/`, ou seja, ~190 GB em disco no total, e as partições de dados ocupam ~1,8 GB (`k = 8`) e ~3,6 GB (`k = 16`) por seed, ~54 GB para 10 seeds. Cada execução leva alguns minutos. A célula de execução confere o espaço livre antes de começar; `APAGAR_PSI = True` apaga o `psi.pkl` de cada execução depois de concluída (a análise não o usa) e deixa o consumo em ~1,6 GB.

Métricas por rodada, contra o grupo real de cada cliente:

| Métrica | Significado |
|---|---|
| nº de clusters | tende a `k` quando `τ` está bem calibrado |
| ARI | 1 = partição igual à real |
| pureza | fração de clientes cujo cluster é dominado pelo grupo deles (singletons têm pureza 1: leia junto com ARI e nº de clusters) |
| cobertura | fração dos clientes já sorteados (e, portanto, com `Ψ`) |
| participantes | clientes sorteados na rodada (com dropout, no máximo os disponíveis) |

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from scipy.stats import wilcoxon
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.cluster import pair_confusion_matrix

PROJETO = Path.cwd()
SCRIPT = "StoCFL-FCC+CFL.py"
assert (PROJETO / SCRIPT).exists(), "abra o notebook a partir de projetos/StoCFL/"
RESULTADOS_JSON = PROJETO / "logs" / "resultados-dropout.json"

# Parâmetros do teste
N = 240                           # nº de clientes (divisível por todo k de KS)
KS = [8, 16]                      # k: grupos reais do Rotated MNIST, rotações em k ângulos igualmente espaçados
ROUNDS = 50                       # nº de rodadas de comunicação
TAU = 0.75                        # limiar de cosseno (--tau). Ver a leitura no fim: para k = 16 é o menor τ que funcionou no relatório de m, k e τ
MU = 0.05                         # só entra no nome da pasta de logs: sem treino, não afeta a clusterização
SEEDS = list(range(10))           # cada seed gera uma nova partição dos dados (~1,8 GB com k = 8 e ~3,6 GB com k = 16)
FRACOES = [1.0, 0.5]              # m: fração dos clientes sorteada por rodada
DROPS = [0.0, 0.05, 0.1]          # d (--drop): probabilidade de um cliente disponível cair por rodada; 0.0 = StoCFL sem dropout
DROP_ROUNDS = 10                  # L_max (--drop_rounds): cada ausência dura L ~ U{1..L_max} rodadas (o mínimo é 1)
REEXECUTAR = False                # True ignora execuções já concluídas e roda de novo
APAGAR_PSI = False                # True apaga o psi.pkl (~1,6 GB) de cada execução concluída; a análise não o usa
DADOS_JSON = False                # True não lê logs nem roda nada: carrega tudo de logs/resultados-dropout.json e só plota (para depois de apagar as execuções)

assert all(N % k == 0 for k in KS) and 0.0 in DROPS and all(0 <= d < 1 for d in DROPS)
assert all(int(N * f) >= 1 for f in FRACOES)


def verdade(k):
    # gabarito: o cliente c pertence ao grupo c * k // N (blocos contíguos de N/k clientes)
    return [c * k // N for c in range(N)]


def nome(d):
    return "sem dropout" if d == 0 else f"d = {d:g}"


# o script só acrescenta o k ao nome das pastas quando ele não é o padrão (4)
K_PADRAO = 4
print("GPU:", torch.cuda.is_available(), "| clientes por rodada:", {f: int(N * f) for f in FRACOES},
      "| disponibilidade média:", {nome(d): round(1 / (1 + d * (DROP_ROUNDS + 1) / 2), 3) for d in DROPS})

## 1. Execução

`executar(k, frac, seed, drop)` roda o script uma vez. Com `drop = 0` o script não recebe `--drop`, e é o StoCFL de sempre. Uma execução já concluída é reaproveitada se o `cluster.pkl` tiver `ROUNDS` partições **e** a linha de argumentos do log tiver os mesmos `n`, `m`, `seed`, `k`, `μ`, `τ`, `train` e `drop`, porque o nome da pasta de logs não traz o `τ` (execuções antigas com outro `τ` não são reaproveitadas).

Os `m` clientes da rodada, o `k` e o `drop` aparecem no nome da pasta (`...-p120-seed0-lambda0.05-k8-drop0.1-r10-time-...`), então execuções com e sem dropout não se misturam.

In [ ]:
def sufixo(k, sep):
    # o script só acrescenta o k aos nomes quando ele não é o padrão
    return "" if k == K_PADRAO else f"{sep}k{k}"


def tag_drop(drop):
    # com dropout o script acrescenta -drop{d}-r{L_max} antes de -time-
    return f"-drop{drop}-r{DROP_ROUNDS}" if drop > 0 else ""


def dados_prontos(k, seed):
    # as partições rotacionadas já foram gravadas? (decide o --process do script)
    d = PROJETO / f"datasets/rotated_mnist_{N}_{seed}{sufixo(k, '_')}"
    return len(list((d / "train").glob("data*.pkl"))) == N and len(list((d / "test").glob("data*.pkl"))) == k


def pastas_de_log(k, m, seed, drop):
    # o nome da pasta traz n, m, seed, mu, k (se não for o padrão) e o dropout, mas não o tau; ordena da mais antiga para a mais nova
    padrao = f"runs-rotated-mnist-n{N}-p{m}-seed{seed}-lambda{MU}{sufixo(k, '-')}{tag_drop(drop)}-time-*"
    return sorted((PROJETO / "logs").glob(padrao), key=lambda p: p.stat().st_mtime)


def args_do_log(pasta):
    # o script registra os argumentos (Namespace) na primeira linha do log principal
    for linha in (pasta / "rotated_mnist.log").read_text().splitlines():
        if "Namespace(" in linha:
            return dict(re.findall(r"(\w+)=([^,()]+)", linha.split("Namespace(", 1)[1]))
    return {}


def concluida(pasta, k, m, seed, drop):
    # execução completa e com os parâmetros pedidos: uma partição por rodada e os mesmos argumentos no log
    try:
        if len(torch.load(pasta / "cluster.pkl", weights_only=False)) != ROUNDS:
            return False
        a = args_do_log(pasta)
        esperado = {"n": N, "com_round": ROUNDS, "num_per_round": m, "mu": MU, "tau": TAU, "train": 0, "seed": seed, "k": k}
        if any(float(a[c]) != v for c, v in esperado.items()):
            return False
        # logs anteriores à flag --drop não têm drop nos argumentos: valem como drop = 0
        return float(a.get("drop", 0.0)) == drop and (drop == 0 or int(a["drop_rounds"]) == DROP_ROUNDS)
    except Exception:
        return False


def pronta(k, frac, seed, drop):
    m = int(N * frac)
    return next((p for p in reversed(pastas_de_log(k, m, seed, drop)) if concluida(p, k, m, seed, drop)), None)


def executar(k, frac, seed, drop):
    m = int(N * frac)
    if not REEXECUTAR and (pasta := pronta(k, frac, seed, drop)) is not None:
        return pasta, None
    cmd = [sys.executable, SCRIPT, "--n", str(N), "--com_round", str(ROUNDS),
           "--num_per_round", str(m), "--mu", str(MU), "--tau", str(TAU),
           "--train", "0", "--seed", str(seed), "--dataset", "mnist", "--k", str(k),
           "--process", "0" if dados_prontos(k, seed) else "1"]
    if drop > 0:
        cmd += ["--drop", str(drop), "--drop_rounds", str(DROP_ROUNDS)]
    (PROJETO / "logs").mkdir(exist_ok=True)
    saida = PROJETO / "logs" / f"pipeline-dropout-k{k}-m{m}-seed{seed}-drop{drop}.txt"
    inicio = time.time()
    with open(saida, "w") as f:
        r = subprocess.run(cmd, cwd=PROJETO, stdout=f, stderr=subprocess.STDOUT)
    duracao = time.time() - inicio
    if r.returncode != 0:
        print(saida.read_text()[-3000:])
        raise RuntimeError(f"falhou: {' '.join(cmd)}")
    # a pasta criada por esta execução é a mais nova com mtime a partir do início
    pasta = next(p for p in reversed(pastas_de_log(k, m, seed, drop)) if p.stat().st_mtime >= inicio - 1)
    return pasta, duracao


def parametros_atuais():
    return {"N": N, "KS": KS, "ROUNDS": ROUNDS, "TAU": TAU, "MU": MU, "SEEDS": SEEDS, "FRACOES": FRACOES, "DROPS": DROPS, "DROP_ROUNDS": DROP_ROUNDS}


def json_com_estes_parametros():
    # o JSON de resultados existe e foi gerado com os parâmetros da célula 1?
    try:
        return json.loads(RESULTADOS_JSON.read_text())["parametros"] == json.loads(json.dumps(parametros_atuais()))
    except (OSError, KeyError, ValueError):
        return False


def particao_de(rot):
    # lista cliente -> cluster vira dicionário cluster -> membros (o inverso de rotulos, da seção 2)
    p = {}
    for c, cid in enumerate(rot):
        p.setdefault(cid, []).append(c)
    return p


def carregar_json():
    # lê logs/resultados-dropout.json: devolve a pasta de cada execução e os dados brutos de cada uma, no formato de ler_execucao
    dados = json.loads(RESULTADOS_JSON.read_text())
    assert dados["parametros"] == json.loads(json.dumps(parametros_atuais())), f"a célula 1 difere do JSON, que foi gerado com {dados['parametros']}"
    execs, lidos = {}, {}
    for e in dados["execucoes"]:
        execs[(e["k"], e["m"], e["seed"], e["drop"])] = e["pasta"]
        lidos[e["pasta"]] = ([particao_de(r) for r in e["rotulos_por_rodada"]], e["sorteados_por_rodada"], e["indisponiveis_por_rodada"])
    assert set(execs) == set(plano) and len(lidos) == len(execs), "o JSON não tem todas as execuções do plano"
    return execs, lidos


plano = [(k, frac, seed, d) for seed in SEEDS for k in KS for frac in FRACOES for d in DROPS]
brutos = {}   # pasta -> (partição por rodada, participantes, indisponíveis): preenchido por ler_execucao (seção 2) ou, com DADOS_JSON, pelo JSON
if DADOS_JSON:
    execucoes, brutos = carregar_json()
    print(f"{len(execucoes)} execuções lidas de {RESULTADOS_JSON.name}, sem logs: nada será executado")
else:
    pendentes = plano if REEXECUTAR else [c for c in plano if pronta(*c) is None]
    if pendentes and not REEXECUTAR and json_com_estes_parametros():
        raise RuntimeError(f"{len(pendentes)} execuções sem logs, mas {RESULTADOS_JSON.name} já tem os resultados desta grade: "
                           "use DADOS_JSON = True para plotar sem rodar nada, ou REEXECUTAR = True para rodar a grade de novo")

    # espaço em disco: psi.pkl de cada execução (N vetores de ~6,5 MB; só o da execução em curso se APAGAR_PSI) + dados das seeds que faltam (~0,23 GB por k)
    psi_gb = N * 6.5e-3
    dados_gb = sum(0.23 * k for k, seed in {(c[0], c[2]) for c in pendentes} if not dados_prontos(k, seed))
    necessario = (psi_gb if APAGAR_PSI else psi_gb * len(pendentes)) + dados_gb
    livre = shutil.disk_usage(PROJETO).free / 1e9
    print(f"{len(plano)} execuções no plano, {len(pendentes)} pendentes; espaço estimado {necessario:.0f} GB, livre {livre:.0f} GB")
    assert livre > 1.1 * necessario, "pouco espaço em disco: reduza SEEDS/KS/FRACOES/DROPS ou use APAGAR_PSI = True"

    execucoes = {}
    for i, (k, frac, seed, drop) in enumerate(plano, start=1):
        pasta, duracao = executar(k, frac, seed, drop)
        if APAGAR_PSI:
            (pasta / "psi.pkl").unlink(missing_ok=True)
        execucoes[(k, frac, seed, drop)] = pasta
        estado = "reaproveitada" if duracao is None else f"{duracao:.0f} s"
        print(f"[{i}/{len(plano)}] k={k} m={frac:.0%} {nome(drop)} seed={seed}: {estado} -> {pasta.name}")


## 2. Métricas por rodada

O `cluster.pkl` guarda, para cada rodada, o dicionário `cluster_id → membros` (clusters mortos ficam com lista vazia). Os participantes de cada rodada vêm das linhas `client id this round` do log (com dropout, só os disponíveis sorteados), e os indisponíveis, do `availability.pkl`. Na tabela final de cada execução:

- **rodada estável:** primeira rodada a partir da qual a partição não muda mais.
- **pares misturados / separados:** da matriz de pares do scikit-learn, sobre os pares de clientes na partição final. *Misturado* = mesmo cluster em grupos reais diferentes (fusão errada, irreversível); *separado* = mesmo grupo real em clusters diferentes (grupo fragmentado).

In [ ]:
def rotulos(particao):
    # dicionário cluster -> membros vira lista cliente -> cluster (formato do adjusted_rand_score)
    r = [None] * N
    for cid, membros in particao.items():
        for c in membros:
            r[c] = cid
    return r


def pureza(particao, gabarito):
    # fração de clientes que pertencem ao grupo real dominante do seu cluster
    return sum(np.bincount([gabarito[c] for c in membros]).max() for membros in particao.values() if membros) / N


def misturados(particao, gabarito):
    # nº de clusters com clientes de mais de um grupo real (fusões erradas, irreversíveis)
    return sum(1 for membros in particao.values() if len({gabarito[c] for c in membros}) > 1)


def rodada_estavel(log):
    # primeira rodada (1-based) a partir da qual a partição não muda mais
    chave = lambda p: frozenset(frozenset(v) for v in p.values() if v)
    final = chave(log[-1])
    r = len(log)
    while r > 1 and chave(log[r - 2]) == final:
        r -= 1
    return r


def ler_execucao(pasta):
    # partição por rodada, clientes sorteados por rodada (log) e indisponíveis por rodada (só com dropout), lidos uma vez dos arquivos da pasta
    # (com DADOS_JSON, brutos já veio do JSON). O JSON da seção 7 guarda exatamente estas três listas
    chave = str(pasta)
    if chave not in brutos:
        pasta = Path(pasta)
        log = torch.load(pasta / "cluster.pkl", weights_only=False)
        participantes = []
        for linha in (pasta / "rotated_mnist.log").read_text().splitlines():
            achou = re.search(r"client id this round \[(.*?)\]", linha)
            if achou:
                participantes.append([int(x) for x in achou.group(1).split(",") if x.strip()])
        arq = pasta / "availability.pkl"
        indisponiveis = torch.load(arq, weights_only=False) if arq.exists() else [[] for _ in log]
        assert len(log) == len(participantes) == len(indisponiveis), f"{pasta.name}: log, participantes e indisponíveis com nº de rodadas diferente"
        brutos[chave] = (log, participantes, indisponiveis)
    return brutos[chave]


def metricas(pasta, k):
    # uma linha por rodada; clusters mortos (lista vazia) não contam
    gabarito = verdade(k)
    log, participantes, indisponiveis = ler_execucao(pasta)
    visto, linhas = set(), []
    for r, particao in enumerate(log):
        visto |= set(participantes[r])
        linhas.append({
            "rodada": r + 1,
            "clusters": sum(1 for v in particao.values() if v),
            "ari": adjusted_rand_score(gabarito, rotulos(particao)),
            "pureza": pureza(particao, gabarito),
            "misturados": misturados(particao, gabarito),
            "cobertura": len(visto) / N,
            "participantes": len(participantes[r]),
            "disponiveis": 1 - len(indisponiveis[r]) / N,
        })
    return pd.DataFrame(linhas), log


tabelas, finais, particoes = [], [], {}
for (k, frac, seed, drop), pasta in execucoes.items():
    df, log = metricas(pasta, k)
    for coluna, valor in (("drop", drop), ("m", frac), ("k", k), ("seed", seed)):
        df.insert(0, coluna, valor)
    tabelas.append(df)
    particoes[(k, frac, seed, drop)] = log[-1]
    # matriz de pares: [[TN, FP], [FN, TP]] com pares ordenados, por isso o // 2
    (_, fp), (fn, _) = pair_confusion_matrix(verdade(k), rotulos(log[-1])) // 2
    u = df.iloc[-1]
    finais.append({"k": k, "m": frac, "drop": drop, "seed": seed, "clusters": int(u["clusters"]), "ari": u["ari"],
                   "pureza": u["pureza"], "misturados": int(u["misturados"]), "estavel": rodada_estavel(log),
                   "cobertura": u["cobertura"], "participantes": df["participantes"].mean(), "disponiveis": df["disponiveis"].mean(),
                   "pares_misturados": int(fp), "pares_separados": int(fn)})
por_rodada = pd.concat(tabelas, ignore_index=True)
finais = pd.DataFrame(finais)

# resumo entre as seeds, uma linha por (k, m, cenário)
resumo = finais.groupby(["k", "m", "drop"]).agg(
    execucoes=("seed", "count"), clusters=("clusters", "mean"), ari=("ari", "mean"), ari_dp=("ari", "std"), ari_min=("ari", "min"),
    seeds_ari_1=("ari", lambda s: int((s > 1 - 1e-9).sum())), seeds_colapso=("clusters", lambda s: int((s == 1).sum())),
    misturados=("misturados", "mean"), estavel=("estavel", "mean"), cobertura=("cobertura", "mean"),
    participantes=("participantes", "mean"), disponiveis=("disponiveis", "mean"),
    pares_misturados=("pares_misturados", "sum"), pares_separados=("pares_separados", "sum")).reset_index()
resumo.insert(2, "cenário", resumo.pop("drop").map(nome))
resumo["m"] = resumo["m"].map(lambda f: f"{f:.0%}")
display(resumo.round(3))

### Dropout × sem dropout, seed a seed

Para cada seed, a diferença `dropout − sem dropout` no ARI final e na rodada estável, com o **teste de Wilcoxon** dos pares. As seeds estão pareadas pela **partição dos dados** (a mesma seed gera os mesmos dados), mas **não pelo sorteio dos clientes**: com `m` < 100% a sequência sorteada diverge assim que alguém cai (plano, seção 2). Com 10 seeds o teste tem pouco poder, e os p-valores são orientativos (24 testes na tabela, 8 linhas × 3 métricas, sem correção para comparações múltiplas).

In [ ]:
linhas = []
for (k, frac), g in finais.groupby(["k", "m"]):
    base = g[g["drop"] == 0].set_index("seed")
    for d in sorted(set(g["drop"]) - {0.0}):
        com = g[g["drop"] == d].set_index("seed")
        seeds = base.index.intersection(com.index)
        linha = {"k": k, "m": f"{frac:.0%}", "cenário": nome(d), "seeds": len(seeds)}
        for col, rotulo in (("ari", "ARI final"), ("estavel", "rodada estável"), ("participantes", "participantes/rodada")):
            dif = (com.loc[seeds, col] - base.loc[seeds, col]).astype(float)
            linha[f"Δ {rotulo}"] = round(dif.mean(), 3)
            # todas as diferenças zero: não há o que testar (p = 1), e o wilcoxon nem é chamado
            linha[f"p ({rotulo})"] = 1.0 if (dif == 0).all() else round(wilcoxon(dif).pvalue, 3)
        linhas.append(linha)
comparacao = pd.DataFrame(linhas)
display(comparacao)

## 3. Gráficos por rodada

Um painel por métrica (linhas) e por `m` (colunas), uma figura por `k`. A linha é a **média entre as seeds** e a faixa vai do **mínimo ao máximo**; clique na legenda para ocultar um cenário e passe o mouse para ver os valores. O painel de participantes mostra o efeito direto do dropout: com `m = 100%`, os participantes caem para o nº de disponíveis.

In [ ]:
CORES_D = ["#2a78d6", "#eb6834", "#1baf7a", "#8a5cd0", "#eda100"]
TRACOS_D = ["solid", "dash", "dot", "dashdot", "longdash"]
cor = lambda d: CORES_D[DROPS.index(d) % len(CORES_D)]
traco = lambda d: TRACOS_D[DROPS.index(d) % len(TRACOS_D)]

paineis = [("clusters", "Nº de clusters"), ("ari", "ARI contra os grupos reais"), ("cobertura", "Clientes já observados"), ("participantes", "Participantes na rodada")]

for k in KS:
    d_k = por_rodada[por_rodada["k"] == k]
    titulos = [f"{t}, m = {f:.0%}" for _, t in paineis for f in FRACOES]
    fig = make_subplots(rows=len(paineis), cols=len(FRACOES), subplot_titles=titulos, horizontal_spacing=0.08, vertical_spacing=0.07)
    for i, (col, _) in enumerate(paineis, start=1):
        for j, frac in enumerate(FRACOES, start=1):
            for d in DROPS:
                g = d_k[(d_k["m"] == frac) & (d_k["drop"] == d)].groupby("rodada")[col]
                media = g.mean()
                leg = f"{nome(d)}"
                fig.add_trace(go.Scatter(x=list(media.index) + list(media.index[::-1]), y=list(g.max()) + list(g.min()[::-1]),
                                         fill="toself", fillcolor=cor(d), opacity=0.12, line=dict(width=0), hoverinfo="skip",
                                         showlegend=False, legendgroup=leg), row=i, col=j)
                fig.add_trace(go.Scatter(x=media.index, y=media.values, mode="lines", name=leg, legendgroup=leg,
                                         showlegend=(i == 1 and j == 1), line=dict(color=cor(d), dash=traco(d), width=2),
                                         hovertemplate="rodada %{x}: %{y:.3g}<extra>" + leg + "</extra>"), row=i, col=j)
            fig.update_xaxes(title_text="rodada" if i == len(paineis) else None, row=i, col=j)
    for j in range(1, len(FRACOES) + 1):
        fig.add_hline(y=k, line=dict(color="gray", width=1, dash="dot"), annotation_text=f"k real = {k}", annotation_position="top right", row=1, col=j)
    fig.update_layout(title=f"k = {k} grupos reais, τ = {TAU}, {len(SEEDS)} seeds (média e faixa mín–máx)", height=230 * len(paineis) + 120,
                      hovermode="x unified", margin=dict(t=90, b=40, l=50, r=20), legend=dict(orientation="h", y=-0.06), template="simple_white")
    fig.update_xaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.show()

## 4. Resultado final por seed

Cada ponto é uma seed; a caixa resume as seeds de cada cenário (mediana, quartis). Um ARI próximo de 0 com 1 cluster é **colapso**: grupos diferentes fundidos de forma irreversível, por `τ` baixo para aquele `k`, e não por dropout, se acontecer também sem dropout (ver a leitura no fim).

In [ ]:
metricas_final = [("ari", "ARI final"), ("clusters", "Nº de clusters finais"), ("estavel", "Rodada estável")]
for k in KS:
    f_k = finais[finais["k"] == k]
    titulos = [f"{t}, m = {f:.0%}" for f in FRACOES for _, t in metricas_final]
    fig = make_subplots(rows=len(FRACOES), cols=len(metricas_final), subplot_titles=titulos, horizontal_spacing=0.07, vertical_spacing=0.14)
    for i, frac in enumerate(FRACOES, start=1):
        for j, (col, _) in enumerate(metricas_final, start=1):
            for d in DROPS:
                v = f_k[(f_k["m"] == frac) & (f_k["drop"] == d)]
                fig.add_trace(go.Box(y=v[col], x=[nome(d)] * len(v), name=nome(d), marker=dict(color=cor(d)), boxpoints="all", jitter=0.4,
                                     pointpos=0, showlegend=(i == 1 and j == 1), legendgroup=nome(d), text=[f"seed {s}" for s in v["seed"]],
                                     hovertemplate="%{text}: %{y:.3g}<extra>" + nome(d) + "</extra>"), row=i, col=j)
    fig.update_layout(title=f"k = {k} grupos reais, τ = {TAU}: uma seed por ponto", height=330 * len(FRACOES) + 100,
                      margin=dict(t=90, b=40, l=50, r=20), legend=dict(orientation="h", y=-0.08), template="simple_white")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.show()

## 5. Matrizes de confusão

Dois olhares sobre a **partição final** (rodada `ROUNDS`), um cenário por painel, uma figura por `k` e `m`:

1. **Grupo real × grupo previsto, somada nas seeds.** O grupo previsto de um cluster é o grupo real dominante dele (empate: o de menor índice). A diagonal é o acerto; fora dela estão os clientes de grupos **vizinhos** (ângulos próximos) fundidos num cluster dominado por outro grupo. Cada linha é normalizada (% dos clientes do grupo real, somando as seeds) e o número de clientes aparece em cada célula. **Um grupo fragmentado em vários clusters continua na diagonal** (cada fragmento tem o mesmo grupo dominante), e um cliente nunca sorteado (cluster unitário) também. Esses casos aparecem no nº de clusters e nos pares separados do resumo acima, não aqui.
2. **Cluster × grupo real de uma seed** (menu para trocar a seed). Cada coluna é um cluster final, ordenado pelo grupo dominante e pelo tamanho, e cada linha é um grupo real. Uma coluna com mais de uma célula preenchida é um cluster **misturado**; vários clusters no mesmo grupo são **fragmentos**.

Os ângulos nos eixos são os do grupo (`360·i/k`).

In [ ]:
def matriz_grupos(particao, k):
    # grupo real × grupo previsto, em que o grupo previsto de um cluster é o grupo real dominante dele
    gabarito = verdade(k)
    M = np.zeros((k, k), dtype=int)
    for membros in particao.values():
        if membros:
            previsto = np.bincount([gabarito[c] for c in membros], minlength=k).argmax()
            for c in membros:
                M[gabarito[c], previsto] += 1
    return M


def contingencia(particao, k):
    # linhas = grupos reais; colunas = clusters vivos, ordenados pelo grupo dominante e pelo tamanho
    gabarito = verdade(k)
    colunas = []
    for cid, membros in particao.items():
        if membros:
            cont = np.bincount([gabarito[c] for c in membros], minlength=k)
            colunas.append((int(cont.argmax()), -len(membros), cid, cont))
    colunas.sort(key=lambda t: t[:3])
    return np.array([t[3] for t in colunas]).T, [t[2] for t in colunas]


matrizes_soma = {}   # (k, m, d) -> matriz grupo real × previsto somada nas seeds, para o JSON do item 7
for k in KS:
    angulos = [f"{360 * i / k:g}°" for i in range(k)]
    for frac in FRACOES:
        # 5.1 grupo real × grupo previsto, somada nas seeds
        fig = make_subplots(rows=1, cols=len(DROPS), subplot_titles=[nome(d) for d in DROPS], horizontal_spacing=0.05)
        for j, d in enumerate(DROPS, start=1):
            M = sum(matriz_grupos(particoes[(k, frac, s, d)], k) for s in SEEDS)
            matrizes_soma[(k, frac, d)] = M
            pct = 100 * M / M.sum(axis=1, keepdims=True)
            fig.add_trace(go.Heatmap(z=pct, x=list(range(k)), y=list(range(k)), colorscale="Blues", zmin=0, zmax=100, showscale=(j == len(DROPS)),
                                     colorbar=dict(title="% do grupo real"), text=M, texttemplate="%{text}" if k <= 8 else "",
                                     customdata=M, hovertemplate="real %{y} → previsto %{x}: %{customdata} clientes (%{z:.1f}%)<extra></extra>"), row=1, col=j)
            fig.update_xaxes(title_text="grupo previsto (cluster → grupo dominante)", tickvals=list(range(k)), ticktext=angulos, tickangle=-45, row=1, col=j)
            fig.update_yaxes(title_text="grupo real" if j == 1 else None, tickvals=list(range(k)), ticktext=angulos, autorange="reversed", row=1, col=j)
        fig.update_layout(title=f"Grupo real × previsto, k = {k}, m = {frac:.0%}, soma de {len(SEEDS)} seeds (rodada {ROUNDS})", height=520,
                          margin=dict(t=90, b=40, l=60, r=20), template="simple_white")
        fig.show()

        # 5.2 cluster × grupo real, uma seed por vez (menu)
        fig = make_subplots(rows=1, cols=len(DROPS), subplot_titles=[nome(d) for d in DROPS], horizontal_spacing=0.05)
        for s in SEEDS:
            for j, d in enumerate(DROPS, start=1):
                Z, ids = contingencia(particoes[(k, frac, s, d)], k)
                fig.add_trace(go.Heatmap(z=Z, x=list(range(1, Z.shape[1] + 1)), y=list(range(k)), customdata=np.array([ids] * k), colorscale="Blues",
                                         zmin=0, zmax=N // k, showscale=(j == len(DROPS)), colorbar=dict(title="clientes"),
                                         visible=(s == SEEDS[0]), hovertemplate="cluster %{customdata}: %{z} clientes do grupo real %{y}<extra></extra>"), row=1, col=j)
        for j in range(1, len(DROPS) + 1):
            fig.update_xaxes(title_text="clusters finais (por grupo dominante)", row=1, col=j)
            fig.update_yaxes(title_text="grupo real" if j == 1 else None, tickvals=list(range(k)), ticktext=angulos, autorange="reversed", row=1, col=j)
        botoes = [dict(label=f"seed {s}", method="update", args=[{"visible": [t_s == s for t_s in SEEDS for _ in DROPS]}]) for s in SEEDS]
        fig.update_layout(title=f"Cluster × grupo real, k = {k}, m = {frac:.0%} (rodada {ROUNDS})", height=520, margin=dict(t=110, b=40, l=60, r=20),
                          updatemenus=[dict(buttons=botoes, direction="down", x=0.0, xanchor="left", y=1.25, yanchor="top")], template="simple_white")
        fig.show()

## 6. Leitura

Rode as células acima e compare, para cada `k` e `m`:

- **Sem dropout × com dropout na rodada estável e na cobertura.** O dropout atrasa a observação dos clientes em poucas rodadas; o esperado é um efeito **pequeno**, visível nas primeiras ~10 rodadas e que some perto da rodada 25 (plano, seção 2). Isso não é bug. Com `m = 100%` o esperado é diferença só no nº de participantes (os disponíveis) e uma rodada estável um pouco mais tardia.
- **Participantes por rodada.** Com `m = 100%`, o nº de participantes com dropout deve ficar perto de `N × disponibilidade` (≈ 188 com `d = 0.05`, ≈ 155 com `d = 0.1`); com `m = 50%` (120 clientes) ele só cai quando há menos de 120 disponíveis.
- **Colapso ou fragmentação iguais com e sem dropout** (ARI ~ 0 com 1 cluster, ou clusters demais) indicam que o problema é `τ` ou `m`, e não o dropout. **Atenção a `τ = 0.75` com `k = 16`:** no relatório de m, k e τ ([relatorio-m-k-tau.md](../../docs/relatorio-m-k-tau.md)), com `k = 16` o `τ` mínimo que recuperou os grupos ficou entre 0.7188 (1 cluster) e 0.75, ou seja, `0.75` está **na borda inferior** da janela, e outras seeds podem fundir grupos vizinhos (transição abrupta e irreversível). Se algumas seeds colapsarem, veja a coluna `seeds_colapso` do resumo e as matrizes de confusão antes de atribuir isso ao dropout. Para `k = 8`, `0.75` está dentro da janela (0.47 a 0.94).
- **Pareamento.** As seeds são pareadas pelos dados, mas não pelo sorteio (a sequência sorteada diverge quando alguém cai). Com 10 seeds, diferenças pequenas podem ser ruído de amostragem: olhe a faixa mín–máx dos gráficos e os pontos por seed.
- **Participação desigual.** Com dropout, quem tem ausências longas participa menos. Isso não afeta a clusterização aqui (`--train 0`), mas afetaria os modelos treinados (`gmodel`, modelos de cluster); este notebook não mede treino.
- **Limites.** Só rotação do MNIST, `n = 240`, uma configuração de `L_max`. O efeito do dropout no ARI não foi medido antes deste notebook (plano, seção 2).

## 7. Resultados em JSON

Grava `logs/resultados-dropout.json` (ignorado pelo git, como o `busca-tau.json`) com tudo que as tabelas e os gráficos usam, para análise fora do notebook e para replotar sem os logs (`DADOS_JSON = True` na célula 1; nesse modo esta célula não grava nada):

- `parametros`: `N`, `KS`, `ROUNDS`, `TAU`, `MU`, `SEEDS`, `FRACOES`, `DROPS`, `DROP_ROUNDS`.
- `execucoes`: uma entrada por execução (`k`, `m`, `seed`, `drop`, a pasta de logs e as métricas finais, com a partição final em `rotulos`, o cluster de cada cliente), mais os dados brutos que bastam para replotar sem os logs: `rotulos_por_rodada` (o cluster de cada cliente em cada rodada), `sorteados_por_rodada` (os clientes sorteados em cada rodada) e `indisponiveis_por_rodada` (os indisponíveis em cada rodada). A chave `participantes` de cada execução continua sendo a média de participantes por rodada. `cossenos_entre_clusters` guarda `[cluster a, cluster b, cosseno]` de cada par de clusters da última rodada (do `cos_rec.log`): o maior valor, comparado com `τ`, é a margem da partição final.
- `por_rodada`: as métricas de cada execução em cada rodada.
- `resumo` e `comparacao`: as duas tabelas das seções 2 e 3 (médias entre as seeds e dropout − sem dropout com Wilcoxon).
- `matrizes_grupo_real_x_previsto`: a matriz de confusão da seção 5.1 (linhas: grupo real; colunas: grupo previsto), somada nas seeds, em contagem de clientes.

In [ ]:
from datetime import datetime


def registros(df):
    # DataFrame -> lista de dicionários só com tipos do Python e sem NaN (JSON estrito)
    return json.loads(df.to_json(orient="records"))


def cossenos_entre_clusters(pasta):
    # cos_rec.log guarda, da última rodada com 2 clusters ou mais, o cosseno de cada par de clusters ("a,b:cos"), do maior para o menor.
    # O maior valor, comparado com TAU, é a margem da partição final: acima de TAU o par teria sido fundido
    arq = Path(pasta) / "cos_rec.log"
    achados = [re.search(r"INFO - (\d+),(\d+):(-?[\d.]+)$", l) for l in arq.read_text().splitlines()] if arq.exists() else []
    return [[int(a), int(b), float(c)] for a, b, c in (m.groups() for m in achados if m)]


def execucao_json(linha):
    # métricas finais + dados brutos: partição de cada rodada (cluster de cada cliente), sorteados e indisponíveis por rodada, e os cossenos finais
    pasta = execucoes[(linha["k"], linha["m"], linha["seed"], linha["drop"])]
    log, participantes, indisponiveis = ler_execucao(pasta)
    rot_rodadas = [[int(r) for r in rotulos(p)] for p in log]
    return {**linha, "pasta": Path(pasta).name, "rotulos": rot_rodadas[-1], "rotulos_por_rodada": rot_rodadas,
            "sorteados_por_rodada": participantes, "indisponiveis_por_rodada": [[int(c) for c in u] for u in indisponiveis],
            "cossenos_entre_clusters": cossenos_entre_clusters(pasta)}


if DADOS_JSON:
    print(f"dados lidos de {RESULTADOS_JSON.name}: nada a gravar")
else:
    resultado = {
        "gerado_em": datetime.now().isoformat(timespec="seconds"),
        "parametros": parametros_atuais(),
        "execucoes": [execucao_json(linha) for linha in registros(finais)],
        "por_rodada": registros(por_rodada),
        "resumo": registros(resumo),
        "comparacao": registros(comparacao),
        "matrizes_grupo_real_x_previsto": [
            {"k": k, "m": m, "drop": d, "seeds": len(SEEDS), "matriz": M.tolist()} for (k, m, d), M in matrizes_soma.items()],
    }
    # grava num arquivo temporário e troca, para nunca deixar o JSON pela metade (ele pode ser a única cópia dos resultados)
    tmp = RESULTADOS_JSON.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(resultado, ensure_ascii=False, allow_nan=False))
    tmp.replace(RESULTADOS_JSON)
    print(f"{RESULTADOS_JSON} ({RESULTADOS_JSON.stat().st_size / 1e6:.2f} MB): {len(resultado['execucoes'])} execuções, {len(resultado['por_rodada'])} linhas por rodada")
